In [1]:
import pandas as pd
import numpy as np

# 1. Load wildfire and airport data
wildfire_df = pd.read_csv("../data/processed_data/Wildfire_Weather_2020_2024_with_gacc.csv")
airport_df = pd.read_csv("../data/processed_data/airports_runways_joined.csv")

# 2. Clean wildfire data
wildfire_df = wildfire_df.rename(columns={
    'startdateyear': 'year',
    'startdatemonth': 'month',
    'startdateday': 'day'
})
wildfire_df['startdate'] = pd.to_datetime(wildfire_df[['year', 'month', 'day']], errors='coerce')
wildfire_df = wildfire_df.dropna(subset=['latitude', 'longitude', 'gacc'])

wildfire_df['centroid_lat'] = wildfire_df['latitude']
wildfire_df['centroid_lon'] = wildfire_df['longitude']

# 3. Clean airport data
airport_df = airport_df.dropna(subset=['latitude_deg', 'longitude_deg', 'gacc'])

# Ensure 'airtanker_base' is boolean
airport_df['airtanker_base'] = airport_df['airtanker_base'].astype(bool)

# 4. Haversine distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a)) * 1000  # meters

# 5. Match wildfires to nearest airport and airtanker base within same GACC
results = []

for _, fire in wildfire_df.iterrows():
    lat1, lon1 = fire['centroid_lat'], fire['centroid_lon']
    fire_gacc = fire['gacc']
    
    # Filter airports to same GACC as wildfire
    airports_in_gacc = airport_df[airport_df['gacc'] == fire_gacc]
    if airports_in_gacc.empty:
        continue
    
    # Distances to all airports in same GACC
    all_distances = haversine(lat1, lon1, airports_in_gacc['latitude_deg'], airports_in_gacc['longitude_deg'])
    all_distances_array = all_distances.values
    
    # Nearest airport (any)
    idx_airport = np.argmin(all_distances_array)
    nearest_airport = airports_in_gacc.iloc[idx_airport]
    
    results.append({
        'fire_id': fire['unique_id'],
        'fire_lat': lat1,
        'fire_lon': lon1,
        'startdate': fire['startdate'],
        'duration': fire['duration'],
        'size (acres)': fire['size (acres)'],
        'fire_spread (acres/day)': fire['fire_spread (acres/day)'],
        'gacc': fire_gacc,
        'distance_nm': all_distances_array[idx_airport] / 1852,
        'ident': nearest_airport['ident'],
        'iata_code': nearest_airport['iata_code'],
        'icao_code': nearest_airport['icao_code'],
        'local_code': nearest_airport['local_code'],
        'closet_airport_name': nearest_airport['name'],
        'type': nearest_airport['type'],
        'latitude_deg': nearest_airport['latitude_deg'],
        'longitude_deg': nearest_airport['longitude_deg'],
        'elevation_ft': nearest_airport['elevation_ft'],
        'country_name': nearest_airport['country_name'],
        'region_name': nearest_airport['region_name'],
        'runway_lengths_ft': nearest_airport['runway_lengths_ft'],
        'runway_surfaces': nearest_airport['runway_surfaces'],
        'airtanker_base': bool(nearest_airport['airtanker_base'])
    })
    
    # Nearest airtanker base within same GACC (could be same as above)
    airtanker_df = airports_in_gacc[airports_in_gacc['airtanker_base'] == True]
    if not airtanker_df.empty:
        base_distances = haversine(lat1, lon1, airtanker_df['latitude_deg'], airtanker_df['longitude_deg'])
        base_distances_array = base_distances.values
        idx_base = np.argmin(base_distances_array)
        nearest_base = airtanker_df.iloc[idx_base]

        results.append({
            'fire_id': fire['unique_id'],
            'fire_lat': lat1,
            'fire_lon': lon1,
            'startdate': fire['startdate'],
            'duration': fire['duration'],
            'size (acres)': fire['size (acres)'],
            'fire_spread (acres/day)': fire['fire_spread (acres/day)'],
            'gacc': fire_gacc,
            'distance_nm': base_distances_array[idx_base] / 1852,
            'ident': nearest_base['ident'],
            'iata_code': nearest_base['iata_code'],
            'icao_code': nearest_base['icao_code'],
            'local_code': nearest_base['local_code'],
            'closet_airport_name': nearest_base['name'],
            'type': nearest_base['type'],
            'latitude_deg': nearest_base['latitude_deg'],
            'longitude_deg': nearest_base['longitude_deg'],
            'elevation_ft': nearest_base['elevation_ft'],
            'country_name': nearest_base['country_name'],
            'region_name': nearest_base['region_name'],
            'runway_lengths_ft': nearest_base['runway_lengths_ft'],
            'runway_surfaces': nearest_base['runway_surfaces'],
            'airtanker_base': True
        })

# 6. Convert to DataFrame and save
result_df = pd.DataFrame(results)
result_df.to_csv("../data/processed_data/nearest_airport_airtanker_bases_to_fires_final.csv", index=False)

print(result_df.head())


      fire_id  fire_lat  fire_lon  startdate  duration  size (acres)  \
0  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
1  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
2  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
3  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
4  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   

   fire_spread (acres/day)                               gacc  distance_nm  \
0                 153.2547  Southern Area Coordination Center    18.608426   
1                 153.2547  Southern Area Coordination Center   403.525444   
2                 153.2547  Southern Area Coordination Center    18.608426   
3                 153.2547  Southern Area Coordination Center   403.525444   
4                 153.2547  Southern Area Coordination Center    18.608426   

  ident  ... closet_airport_name            type latitude_deg longitude_deg  \
0  KCEW  ...   Bob 

In [3]:
# === CHECKER CODE ===
# Check if any known airtanker base airport appears in results but marked as False

# List of known airtanker base idents in airports dataset for checking
known_airtanker_bases = airport_df[airport_df['airtanker_base'] == True]['ident'].unique()

# Filter results for rows where airport is in known airtanker bases but airtanker_base is False
incorrect_flags = result_df[
    (result_df['ident'].isin(known_airtanker_bases)) & (result_df['airtanker_base'] == False)
]

if not incorrect_flags.empty:
    print("Warning: The following airport records are known airtanker bases but marked False in results:")
    print(incorrect_flags[['fire_id', 'ident', 'closet_airport_name', 'airtanker_base']])
else:
    print("All known airtanker bases correctly marked in results.")

print("\nSample of result data:")
print(result_df.head())

All known airtanker bases correctly marked in results.

Sample of result data:
      fire_id  fire_lat  fire_lon  startdate  duration  size (acres)  \
0  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
1  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
2  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
3  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
4  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   

   fire_spread (acres/day)                               gacc  distance_nm  \
0                 153.2547  Southern Area Coordination Center    18.608426   
1                 153.2547  Southern Area Coordination Center   403.525444   
2                 153.2547  Southern Area Coordination Center    18.608426   
3                 153.2547  Southern Area Coordination Center   403.525444   
4                 153.2547  Southern Area Coordination Center    18.608426   

  ident  ... closet